# BCH 디코더의 신드롬 계산 완벽 가이드

이 노트북은 BCH 디코딩의 첫 번째 단계인 **신드롬 계산(Syndrome Computation)**을 심층적으로 학습하기 위한 것입니다.

## 목차
1. [신드롬이란 무엇인가?](#1.-신드롬이란-무엇인가?)
2. [신드롬의 수학적 정의](#2.-신드롬의-수학적-정의)
3. [계산 방법 1: 다항식 평가](#3.-계산-방법-1:-다항식-평가)
4. [계산 방법 2: 다항식 나눗셈](#4.-계산-방법-2:-다항식-나눗셈)
5. [Horner 방법 상세 분석](#5.-Horner-방법-상세-분석)
6. [GF(2^m) 참조 테이블](#6.-GF(2^m)-참조-테이블)
7. [신드롬 시각화](#7.-신드롬-시각화)
8. [대화형 실험](#8.-대화형-실험)
9. [두 방법 비교](#9.-두-방법-비교)
10. [신드롬과 오류의 관계](#10.-신드롬과-오류의-관계)
11. [BCH 디코더에서의 역할](#11.-BCH-디코더에서의-역할)
12. [하드웨어 구현 고려사항](#12.-하드웨어-구현-고려사항)

In [ ]:
# 필요한 모듈 임포트
import sys
sys.path.append('..')

from bch_learning import (
    GaloisField, GFElement,
    BCHCode
)
from bch_learning.syndrome_calculator import (
    SyndromeCalculator,
    create_gf_reference_table,
    test_syndrome_with_errors
)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches

# 한글 폰트 설정 (필요시)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

## 1. 신드롬이란 무엇인가?

### 개념

**신드롬(Syndrome)**은 수신된 코드워드가 유효한 코드워드인지 확인하는 "검사 값"입니다.

### 의료 용어와의 유사성

의학에서 "증후군(syndrome)"은 질병의 징후와 증상의 집합입니다. BCH 코드에서도 마찬가지로:
- **신드롬 = 0**: 건강한 상태 (오류 없음)
- **신드롬 ≠ 0**: 질병 상태 (오류 존재)
- **신드롬 패턴**: 어떤 종류의 오류인지 알려줌

### BCH 코드에서의 역할

```
송신: 유효한 코드워드 c(x)
  ↓ (채널을 통해 전송)
수신: r(x) = c(x) + e(x)  (여기서 e(x)는 오류)
  ↓
신드롬 계산: S_i = r(α^i)
  ↓
신드롬 = 0? → 오류 없음
신드롬 ≠ 0? → 오류 존재, 다음 단계로
```

### 왜 신드롬을 계산하는가?

1. **오류 감지**: 신드롬이 0이 아니면 오류가 있음을 즉시 알 수 있음
2. **오류 위치 정보**: 신드롬 패턴이 오류 위치에 대한 정보를 담고 있음
3. **효율성**: 전체 코드워드를 검사하지 않고도 오류 확인 가능

## 2. 신드롬의 수학적 정의

### BCH 코드의 기본 성질

BCH(n, k, t) 코드의 생성 다항식 g(x)는 α, α^2, ..., α^(2t)를 근으로 가집니다.

따라서 **모든 유효한 코드워드** c(x)에 대해:

$$c(\alpha^i) = 0 \quad \text{for } i = 1, 2, \ldots, 2t$$

### 신드롬 정의

수신된 워드 r(x)에 대해 신드롬은:

$$S_i = r(\alpha^i) \quad \text{for } i = 1, 2, \ldots, 2t$$

### 오류 다항식

오류를 e(x) = r(x) - c(x)라 하면 (GF(2)에서 - = +):

$$S_i = r(\alpha^i) = c(\alpha^i) + e(\alpha^i) = 0 + e(\alpha^i) = e(\alpha^i)$$

**중요**: 신드롬은 오류 다항식의 값과 같습니다!

### 예시: 단일 오류

위치 j에 단일 오류가 있다면:
$$e(x) = x^j$$
$$S_i = e(\alpha^i) = (\alpha^i)^j = \alpha^{ij}$$

### 예시: 다중 오류

위치 j_1, j_2, ..., j_v에 오류가 있다면:
$$e(x) = x^{j_1} + x^{j_2} + \cdots + x^{j_v}$$
$$S_i = \alpha^{ij_1} + \alpha^{ij_2} + \cdots + \alpha^{ij_v}$$

In [ ]:
# GF(2^4) 생성
gf = GaloisField(4)
print(f"생성된 유한체: {gf}")
print(f"코드 길이: n = 2^{gf.m} - 1 = {gf.order}")
print(f"원시 다항식: {bin(gf.primitive_poly)}")

## 3. 계산 방법 1: 다항식 평가

### 원리

수신 워드를 다항식으로 표현:
$$r(x) = r_0 + r_1 x + r_2 x^2 + \cdots + r_{n-1} x^{n-1}$$

α^i를 대입하여 평가:
$$S_i = r(\alpha^i) = r_0 + r_1 \alpha^i + r_2 \alpha^{2i} + \cdots + r_{n-1} \alpha^{(n-1)i}$$

### Horner 방법

효율적인 계산을 위해 Horner 방법 사용:

$$r(x) = r_0 + x(r_1 + x(r_2 + x(\cdots + x(r_{n-2} + x \cdot r_{n-1})\cdots)))$$

알고리즘:
```
result = 0
for i from n-1 down to 0:
    result = result × α^i + r_i
return result
```

### 계산 복잡도

- n번의 GF 곱셈
- n번의 GF 덧셈
- 총 O(n) 연산

In [ ]:
# 신드롬 계산기 생성
calc = SyndromeCalculator(gf, t=1, verbose=True)

# 예시: 위치 5에 오류가 있는 경우
print("=" * 80)
print("예시 1: 위치 5에 단일 오류")
print("=" * 80)

received = [0] * 15
received[5] = 1

print(f"수신 벡터: {''.join(map(str, received))}")
print(f"오류 위치: 5")
print(f"예상 신드롬: S_1 = α^5, S_2 = α^10")

result = calc.compute_by_evaluation(received)

## 4. 계산 방법 2: 다항식 나눗셈

### 다항식 나머지 정리

다항식 r(x)를 (x - α^i)로 나눈 나머지는 r(α^i)입니다:

$$r(x) = q(x) \cdot (x - \alpha^i) + r(\alpha^i)$$

따라서:
$$S_i = r(\alpha^i) = \text{remainder of } r(x) / (x - \alpha^i)$$

### GF(2)에서의 나눗셈

GF(2)에서 나눗셈은 XOR 연산으로 수행됩니다:

```
         _____________
divisor | dividend
        | XOR
        | ─────
        | 중간 결과
        | ...
        | ─────
        | 나머지 (신드롬)
```

### 실제 구현

실제로는 r(x)를 각 (x - α^i)로 나누는 것보다
직접 평가하는 것이 더 효율적입니다.

하지만 개념적 이해를 위해 나눗셈 관점을 아는 것이 중요합니다.

### 하드웨어 구현

하드웨어에서는 **선형 피드백 시프트 레지스터(LFSR)**를 사용하여
나눗셈을 효율적으로 수행할 수 있습니다.

## 5. Horner 방법 상세 분석

### 왜 Horner 방법을 사용하는가?

**직접 계산**:
$$r(\alpha^i) = r_0 + r_1 \alpha^i + r_2 \alpha^{2i} + r_3 \alpha^{3i} + \cdots$$

- n번의 거듭제곱 연산 필요 (α^i, α^{2i}, α^{3i}, ...)
- 각 거듭제곱마다 여러 번의 곱셈
- 총 O(n^2) 연산

**Horner 방법**:
$$r(\alpha^i) = r_0 + \alpha^i \cdot (r_1 + \alpha^i \cdot (r_2 + \alpha^i \cdot (\cdots)))$$

- n번의 곱셈만 필요
- 총 O(n) 연산
- **훨씬 효율적!**

In [ ]:
# Horner 방법 상세 추적
print("=" * 80)
print("Horner 방법 상세 분석")
print("=" * 80)

# 간단한 예: 위치 3에 오류
received = [0] * 15
received[3] = 1

print(f"\n수신 벡터: {''.join(map(str, received))}")
print(f"오류 위치: 3")
print(f"\n다항식 표현: r(x) = x^3")
print(f"예상: S_1 = α^3, S_2 = α^6")

# t=1이므로 S_1, S_2 계산
calc_detail = SyndromeCalculator(gf, t=1, verbose=True)
result_detail = calc_detail.compute_by_evaluation(received)

### Horner 방법 단계별 표

위의 계산에서 각 단계를 살펴보면:

| 반복 | 계수 | 현재값 | 연산 | 다음값 |
|------|------|--------|------|--------|
| 1 | r_14 | 0 | 초기값 | r_14 |
| 2 | r_13 | r_14 | r_14 × α + r_13 | ... |
| ... | ... | ... | ... | ... |
| 15 | r_0 | ... | ... × α + r_0 | S_1 |

이렇게 순차적으로 계산하면 최종적으로 S_1을 얻습니다.

## 6. GF(2^m) 참조 테이블

신드롬 계산을 이해하려면 GF(2^m)의 모든 원소를 알아야 합니다.

In [ ]:
# GF(2^4) 완전한 참조 테이블
create_gf_reference_table(gf)

### GF 연산 예시

In [ ]:
print("=" * 60)
print("GF(2^4) 연산 예시")
print("=" * 60)

# 예시 1: 덧셈 (XOR)
a = gf.alpha(3)  # α^3
b = gf.alpha(5)  # α^5
c = a + b

print(f"\n덧셈 (XOR):")
print(f"  {a} (이진: {a.to_binary()})")
print(f"+ {b} (이진: {b.to_binary()})")
print(f"= {c} (이진: {c.to_binary()})")
print(f"  XOR: {a.poly:04b} XOR {b.poly:04b} = {c.poly:04b}")

# 예시 2: 곱셈 (지수 덧셈)
d = a * b

print(f"\n곱셈 (지수 덧셈):")
print(f"  {a} × {b}")
print(f"= α^(3+5) = α^8")
print(f"= {d}")

# 예시 3: 거듭제곱
e = a ** 5

print(f"\n거듭제곱:")
print(f"  ({a})^5 = α^(3×5) = α^15 = α^0 = {e}")

## 7. 신드롬 시각화

### 신드롬을 이진 벡터로 표현

In [ ]:
def visualize_syndromes_binary(syndromes, title="신드롬 이진 표현"):
    """
    신드롬을 이진 벡터로 시각화
    """
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # 각 신드롬을 이진수로 변환
    num_syndromes = len(syndromes)
    m = syndromes[0].field.m if syndromes else 4
    
    # 비트 배열 생성
    for i, s in enumerate(syndromes):
        binary = s.to_binary() if not s.is_zero() else '0' * m
        
        for j, bit in enumerate(binary):
            color = 'red' if bit == '1' else 'lightgray'
            rect = Rectangle((j, num_syndromes - i - 1), 1, 0.8, 
                            facecolor=color, edgecolor='black')
            ax.add_patch(rect)
            
            # 비트 값 표시
            ax.text(j + 0.5, num_syndromes - i - 0.5, bit,
                   ha='center', va='center', fontsize=14, fontweight='bold')
    
    # 축 설정
    ax.set_xlim(0, m)
    ax.set_ylim(0, num_syndromes)
    ax.set_xticks(np.arange(m) + 0.5)
    ax.set_xticklabels([f'b{i}' for i in range(m-1, -1, -1)])
    ax.set_yticks(np.arange(num_syndromes) + 0.5)
    ax.set_yticklabels([f'S_{num_syndromes - i}' for i in range(num_syndromes)])
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('비트 위치', fontsize=12)
    
    plt.tight_layout()
    plt.show()

# 예시: 위치 5에 오류
received_vis = [0] * 15
received_vis[5] = 1

calc_vis = SyndromeCalculator(gf, t=2, verbose=False)
result_vis = calc_vis.compute_by_evaluation(received_vis)

visualize_syndromes_binary(result_vis.syndromes, "위치 5 오류의 신드롬")

### 코드워드 → 신드롬 흐름 시각화

In [ ]:
def visualize_codeword_to_syndrome(received, syndromes, error_pos):
    """
    코드워드에서 신드롬까지의 흐름 시각화
    """
    fig, axes = plt.subplots(3, 1, figsize=(14, 8))
    
    # 1. 수신 코드워드
    ax1 = axes[0]
    for i, bit in enumerate(received):
        color = 'red' if i in error_pos else ('green' if bit == 1 else 'lightgray')
        rect = Rectangle((i, 0), 1, 0.8, facecolor=color, edgecolor='black')
        ax1.add_patch(rect)
        ax1.text(i + 0.5, 0.4, str(bit), ha='center', va='center', 
                fontsize=10, fontweight='bold')
    
    ax1.set_xlim(0, len(received))
    ax1.set_ylim(0, 1)
    ax1.set_title('수신 코드워드 (빨강: 오류 위치)', fontsize=12, fontweight='bold')
    ax1.set_xticks(np.arange(len(received)) + 0.5)
    ax1.set_xticklabels([str(i) for i in range(len(received))], fontsize=8)
    ax1.set_yticks([])
    
    # 2. 화살표
    ax2 = axes[1]
    ax2.text(0.5, 0.5, '↓\n신드롬 계산\nS_i = R(α^i)', 
            ha='center', va='center', fontsize=14, fontweight='bold')
    ax2.set_xlim(0, 1)
    ax2.set_ylim(0, 1)
    ax2.axis('off')
    
    # 3. 신드롬
    ax3 = axes[2]
    m = syndromes[0].field.m
    
    for i, s in enumerate(syndromes):
        binary = s.to_binary() if not s.is_zero() else '0' * m
        
        for j, bit in enumerate(binary):
            color = 'orange' if bit == '1' else 'lightgray'
            rect = Rectangle((i * (m + 1) + j, 0), 1, 0.8, 
                           facecolor=color, edgecolor='black')
            ax3.add_patch(rect)
            ax3.text(i * (m + 1) + j + 0.5, 0.4, bit, 
                    ha='center', va='center', fontsize=10, fontweight='bold')
        
        # 신드롬 라벨
        ax3.text(i * (m + 1) + m/2, -0.3, f'S_{i+1}={s}', 
                ha='center', va='top', fontsize=10)
    
    ax3.set_xlim(0, len(syndromes) * (m + 1))
    ax3.set_ylim(-0.5, 1)
    ax3.set_title('신드롬 (주황: 1 비트)', fontsize=12, fontweight='bold')
    ax3.set_xticks([])
    ax3.set_yticks([])
    
    plt.tight_layout()
    plt.show()

# 시각화
visualize_codeword_to_syndrome(received_vis, result_vis.syndromes, [5])

### 오류 유무에 따른 신드롬 비교

In [ ]:
def compare_syndrome_with_without_error(bch_code):
    """
    오류 있음 vs 없음 신드롬 비교
    """
    # 유효한 코드워드 생성
    message = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
    codeword = bch_code.encode(message)
    
    # 오류 없음
    syn_no_error = bch_code.compute_syndromes(codeword)
    
    # 오류 있음 (위치 7)
    received = codeword[:]
    received[7] ^= 1
    syn_with_error = bch_code.compute_syndromes(received)
    
    # 비교 표
    print("=" * 60)
    print("오류 유무에 따른 신드롬 비교")
    print("=" * 60)
    print(f"코드워드: {''.join(map(str, codeword))}")
    print(f"수신 (오류 위치 7): {''.join(map(str, received))}")
    print()
    
    print(f"{'신드롬':<10} {'오류 없음':<20} {'오류 있음':<20}")
    print("-" * 50)
    
    for i in range(len(syn_no_error)):
        s_no = syn_no_error[i]
        s_yes = syn_with_error[i]
        
        no_str = "0" if s_no.is_zero() else str(s_no)
        yes_str = "0" if s_yes.is_zero() else str(s_yes)
        
        print(f"S_{i+1:<8} {no_str:<20} {yes_str:<20}")
    
    print("=" * 60)
    print(f"오류 감지: {'아니오' if all(s.is_zero() for s in syn_no_error) else '예'}")
    print(f"오류 감지: {'아니오' if all(s.is_zero() for s in syn_with_error) else '예'}")
    print("=" * 60)

# BCH(15, 11, 1) 코드
bch = BCHCode(m=4, t=1, field=gf)
compare_syndrome_with_without_error(bch)

## 8. 대화형 실험

이제 직접 파라미터를 선택하여 신드롬 계산을 실험해봅시다!

In [ ]:
def interactive_syndrome_experiment(
    bch_type="BCH(15,11,1)",
    message=None,
    error_positions=None,
    random_errors=False
):
    """
    대화형 신드롬 계산 실험
    
    Args:
        bch_type: "BCH(15,11,1)", "BCH(15,7,2)", "BCH(31,21,2)"
        message: 정보 비트 (None이면 랜덤)
        error_positions: 오류 위치 리스트 (None이면 랜덤)
        random_errors: True면 랜덤 오류 생성
    """
    print("\n" + "=" * 80)
    print(f"대화형 신드롬 계산 실험: {bch_type}")
    print("=" * 80)
    
    # BCH 코드 설정
    if bch_type == "BCH(15,11,1)":
        m, t = 4, 1
        field = GaloisField(4)
    elif bch_type == "BCH(15,7,2)":
        m, t = 4, 2
        field = GaloisField(4)
    elif bch_type == "BCH(31,21,2)":
        m, t = 5, 2
        field = GaloisField(5)
    else:
        raise ValueError(f"지원하지 않는 BCH 타입: {bch_type}")
    
    bch = BCHCode(m=m, t=t, field=field)
    
    # 메시지 생성
    if message is None:
        message = [np.random.randint(0, 2) for _ in range(bch.k)]
    
    print(f"\n[1단계] 정보 비트 ({bch.k}비트)")
    print(f"  이진: {''.join(map(str, message))}")
    print(f"  16진: 0x{int(''.join(map(str, message)), 2):X}")
    
    # 인코딩
    codeword = bch.encode(message)
    print(f"\n[2단계] 인코딩 → 코드워드 ({bch.n}비트)")
    print(f"  이진: {''.join(map(str, codeword))}")
    print(f"  16진: 0x{int(''.join(map(str, codeword)), 2):X}")
    
    # 오류 추가
    if error_positions is None:
        if random_errors:
            num_errors = np.random.randint(0, t + 1)
            error_positions = np.random.choice(bch.n, num_errors, replace=False).tolist()
        else:
            error_positions = []
    
    received = bch.add_errors(codeword, error_positions)
    
    print(f"\n[3단계] 오류 삽입")
    print(f"  오류 위치: {error_positions if error_positions else '없음'}")
    print(f"  수신: {''.join(map(str, received))}")
    print(f"  원본: {''.join(map(str, codeword))}")
    if error_positions:
        print(f"  차이: {''.join(['^' if c != r else ' ' for c, r in zip(codeword, received)])}")
    
    # 신드롬 계산 (두 가지 방법)
    print(f"\n[4단계] 신드롬 계산")
    print("─" * 80)
    
    calc = SyndromeCalculator(field, t, verbose=False)
    result = calc.compute_by_evaluation(received)
    
    # 신드롬 출력
    print(f"\n{'신드롬':<10} {'값':<15} {'이진':<10} {'16진':<8}")
    print("─" * 45)
    
    for i, s in enumerate(result.syndromes, 1):
        if s.is_zero():
            print(f"S_{i:<8} {'0':<15} {'0000':<10} {'0x0':<8}")
        else:
            print(f"S_{i:<8} {str(s):<15} {s.to_binary():<10} 0x{s.poly:X:<6}")
    
    # 신드롬 해석
    print(f"\n[5단계] 신드롬 해석")
    if result.is_valid_codeword:
        print("  ✓ 모든 신드롬 = 0")
        print("  → 오류 없음 (또는 감지 불가능한 오류)")
    else:
        print("  ✗ 신드롬 ≠ 0")
        print("  → 오류 존재 감지!")
        print("  → 다음 단계: Berlekamp-Massey 알고리즘으로 오류 위치 찾기")
    
    print("\n" + "=" * 80)
    
    return result

# 실험 1: 오류 없음
print("\n" + "#" * 80)
print("실험 1: 오류 없는 경우")
print("#" * 80)
interactive_syndrome_experiment(
    bch_type="BCH(15,11,1)",
    message=[1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1],
    error_positions=[]
)

In [ ]:
# 실험 2: 단일 오류
print("\n" + "#" * 80)
print("실험 2: 단일 오류 (위치 5)")
print("#" * 80)
interactive_syndrome_experiment(
    bch_type="BCH(15,11,1)",
    message=[1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1],
    error_positions=[5]
)

In [ ]:
# 실험 3: 이중 오류
print("\n" + "#" * 80)
print("실험 3: 이중 오류 (위치 3, 10)")
print("#" * 80)
interactive_syndrome_experiment(
    bch_type="BCH(15,7,2)",
    message=[1, 0, 1, 1, 0, 1, 0],
    error_positions=[3, 10]
)

In [ ]:
# 실험 4: 랜덤
print("\n" + "#" * 80)
print("실험 4: 랜덤 메시지 및 오류")
print("#" * 80)
interactive_syndrome_experiment(
    bch_type="BCH(15,7,2)",
    random_errors=True
)

## 9. 두 방법 비교

### 계산 복잡도 비교

| 방법 | 곱셈 횟수 | 덧셈 횟수 | 총 복잡도 | 비고 |
|------|-----------|-----------|-----------|------|
| 직접 평가 | O(n²) | O(n) | O(n²) | 각 항마다 거듭제곱 |
| Horner 방법 | O(n) | O(n) | O(n) | 효율적! |
| 다항식 나눗셈 | O(n) | O(n) | O(n) | LFSR로 구현 가능 |

### 하드웨어 구현 관점

#### 다항식 평가 (병렬 구조)
```
r_0 ─┬─────────┐
     │         ├─→ + ─→ S_i
r_1 ─┼─ ×α^i ─┤
     │         │
r_2 ─┼─ ×α^2i ┤
     │         │
...  │   ...   │
```
- 모든 항을 병렬로 계산
- 빠르지만 많은 곱셈기 필요
- 면적 ↑, 속도 ↑

#### 다항식 나눗셈 (LFSR 구조)
```
입력 ─→ [D] ─→ [D] ─→ [D] ─→ ...
         ↓      ↓      ↓
         ⊕ ←─ × ←─ × ←─ (피드백)
```
- 순차적으로 계산
- 적은 하드웨어 자원
- 면적 ↓, 속도 ↓

In [ ]:
# 실제 계산 시간 비교 (Python에서는 큰 차이 없음)
import time

def benchmark_syndrome_methods(field, t, num_trials=1000):
    """
    신드롬 계산 방법 벤치마크
    """
    calc = SyndromeCalculator(field, t, verbose=False)
    n = field.order
    
    # 랜덤 수신 워드 생성
    test_vectors = []
    for _ in range(num_trials):
        received = [np.random.randint(0, 2) for _ in range(n)]
        test_vectors.append(received)
    
    # 평가 방법
    start = time.time()
    for received in test_vectors:
        calc.compute_by_evaluation(received)
    time_eval = time.time() - start
    
    print(f"\n벤치마크 결과 ({num_trials}회 시행)")
    print("─" * 40)
    print(f"다항식 평가 방법: {time_eval:.4f}초")
    print(f"평균 시간: {time_eval/num_trials*1000:.3f}ms")

benchmark_syndrome_methods(gf, t=2, num_trials=100)

## 10. 신드롬과 오류의 관계

### 신드롬 패턴 분석

In [ ]:
# 각 위치의 오류에 대한 신드롬 패턴
calc_pattern = SyndromeCalculator(gf, t=2, verbose=False)

print("=" * 80)
print("단일 오류 위치별 신드롬 패턴")
print("=" * 80)
print(f"{'위치':<6} {'S_1':<12} {'S_2':<12} {'S_3':<12} {'S_4':<12}")
print("─" * 60)

for pos in range(15):
    received = [0] * 15
    received[pos] = 1
    
    result = calc_pattern.compute_by_evaluation(received)
    syns = result.syndromes
    
    print(f"{pos:<6} {str(syns[0]):<12} {str(syns[1]):<12} {str(syns[2]):<12} {str(syns[3]):<12}")

print("\n관찰:")
print("  - 위치 j의 오류 → S_i = α^(ij)")
print("  - S_2 = (S_1)^2 관계 (단일 오류)")
print("  - S_4 = (S_2)^2 = (S_1)^4 관계 (단일 오류)")

### 다중 오류의 신드롬

In [ ]:
print("\n=" * 80)
print("다중 오류의 신드롬")
print("=" * 80)

# 위치 3과 10에 오류
pos1, pos2 = 3, 10

# 개별 오류의 신드롬
r1 = [0] * 15
r1[pos1] = 1
s1_individual = calc_pattern.compute_by_evaluation(r1).syndromes

r2 = [0] * 15
r2[pos2] = 1
s2_individual = calc_pattern.compute_by_evaluation(r2).syndromes

# 두 오류의 합
r_both = [0] * 15
r_both[pos1] = 1
r_both[pos2] = 1
s_both = calc_pattern.compute_by_evaluation(r_both).syndromes

print(f"위치 {pos1} 오류: {[str(s) for s in s1_individual]}")
print(f"위치 {pos2} 오류: {[str(s) for s in s2_individual]}")
print(f"두 오류 합:     {[str(s) for s in s_both]}")

# 검증: 신드롬의 합 = 합의 신드롬
print("\n검증: 신드롬의 선형성")
for i in range(4):
    s_sum = s1_individual[i] + s2_individual[i]
    match = "✓" if s_sum == s_both[i] else "✗"
    print(f"  S_{i+1}: {s1_individual[i]} + {s2_individual[i]} = {s_sum} {'=' if match == '✓' else '≠'} {s_both[i]} {match}")

## 11. BCH 디코더에서의 역할

### 완전한 BCH 디코딩 흐름

```
┌─────────────────────────────────────────┐
│  수신 코드워드 R(x)                      │
└─────────────┬───────────────────────────┘
              │
              ▼
┌─────────────────────────────────────────┐
│  [1단계] 신드롬 계산 ← 현재 학습 중!    │
│  S_i = R(α^i) for i=1,...,2t            │
└─────────────┬───────────────────────────┘
              │
              ├─ 모든 S_i = 0? ─→ 오류 없음, 종료
              │
              ▼ S_i ≠ 0
┌─────────────────────────────────────────┐
│  [2단계] Berlekamp-Massey 알고리즘      │
│  신드롬 → 오류 위치 다항식 Λ(x)         │
└─────────────┬───────────────────────────┘
              │
              ▼
┌─────────────────────────────────────────┐
│  [3단계] Chien Search                    │
│  Λ(x)의 근 찾기 → 오류 위치             │
└─────────────┬───────────────────────────┘
              │
              ▼
┌─────────────────────────────────────────┐
│  오류 정정                               │
│  r_i ← r_i ⊕ 1 (GF(2))                  │
└─────────────┬───────────────────────────┘
              │
              ▼
┌─────────────────────────────────────────┐
│  정정된 코드워드                         │
└─────────────────────────────────────────┘
```

### 신드롬의 중요성

1. **게이트키퍼**: 오류가 없으면 즉시 종료 → 계산량 절감
2. **정보 압축**: n비트 코드워드 → 2t개의 GF(2^m) 원소로 압축
3. **오류 정보**: 다음 단계(BM 알고리즘)의 입력

### 계산 비용

전체 BCH 디코딩에서 신드롬 계산이 차지하는 비중:
- 시간: 약 20-30%
- 하드웨어: LFSR로 구현 시 매우 작음

## 12. 하드웨어 구현 고려사항

### LFSR 기반 신드롬 계산

#### 원리

선형 피드백 시프트 레지스터(LFSR)는 다항식 나눗셈을 하드웨어로 구현한 것입니다.

```
입력 비트 스트림 (r_{n-1}, r_{n-2}, ..., r_0)
       ↓
    ┌──┴──┐
    │ XOR │←─────┐
    └──┬──┘      │
       ↓          │
    ┌─────┐   ┌─────┐   ┌─────┐
    │ D0  │→ │ D1  │→ │ D2  │→ ...
    └─────┘   └─────┘   └─────┘
       ↓         ↓         ↓
      ×g_0     ×g_1     ×g_2
       └─────────┴─────────┘
              (피드백)
```

#### GF(2^m) 곱셈 구현

GF(2^m)에서 α와의 곱셈은 XOR 게이트로 구현:
```
a = (a_{m-1}, ..., a_1, a_0)
a × α = (a_{m-2}, ..., a_0, 0) ⊕ (a_{m-1} × p)
```
여기서 p는 원시 다항식

### 최적화 기법

1. **파이프라인**: 여러 신드롬을 병렬로 계산
2. **룩업 테이블**: 작은 m에 대해 GF 연산을 테이블로 저장
3. **재사용**: 한 번 계산한 α^i를 다른 신드롬에도 사용

### 면적-속도 트레이드오프

| 구조 | 면적 | 속도 | 용도 |
|------|------|------|------|
| 직렬 LFSR | 소 | 저 | 저전력 장치 |
| 병렬 평가 | 대 | 고 | 고속 통신 |
| 하이브리드 | 중 | 중 | 범용 |

### 실제 IP 구현 예시

- **Wi-Fi (802.11)**: BCH(127, 113, 2) 사용
- **QR 코드**: BCH 코드 변형 사용
- **NAND 플래시**: BCH(512, 502, t=4-8) 사용
- **위성 통신**: 긴 BCH 코드 (n > 1000)

대부분 LFSR 기반 구현 사용 (면적 효율성)

## 종합 연습 문제

### 문제 1: 신드롬 계산 손으로 해보기

GF(2^3)에서 (원시 다항식: x^3 + x + 1)
- 코드워드: 0101010
- 위치 2에 오류 발생
- S_1과 S_2를 손으로 계산해보세요

### 문제 2: 신드롬 패턴 분석

다음 신드롬이 주어졌을 때:
- S_1 = α^5
- S_2 = α^10

이것이 단일 오류인지 판단하고, 만약 그렇다면 오류 위치를 추론하세요.

### 문제 3: 구현 최적화

BCH(255, 239, 2) 코드의 신드롬 계산을 최적화하려고 합니다.
하드웨어 면적이 제한되어 있을 때 어떤 구조를 선택하시겠습니까?

In [ ]:
# 문제 1 풀이 영역
print("문제 1 풀이")
print("=" * 60)

gf3 = GaloisField(3)
calc3 = SyndromeCalculator(gf3, t=1, verbose=True)

received_q1 = [0, 1, 0, 1, 0, 1, 0]
# 위치 2에 오류
received_q1[2] ^= 1

result_q1 = calc3.compute_by_evaluation(received_q1)

## 요약 및 다음 단계

### 이번 노트북에서 배운 것

1. ✓ 신드롬의 정의와 의미
2. ✓ 두 가지 계산 방법 (평가 vs 나눗셈)
3. ✓ Horner 방법의 효율성
4. ✓ GF(2^m) 연산과 신드롬의 관계
5. ✓ 신드롬 패턴과 오류의 관계
6. ✓ 하드웨어 구현 고려사항

### 핵심 요점

- **신드롬 = 0**: 오류 없음
- **신드롬 ≠ 0**: 오류 존재
- **S_i = e(α^i)**: 신드롬은 오류 다항식의 값
- **Horner 방법**: O(n) 시간에 효율적 계산
- **LFSR**: 하드웨어로 구현 가능

### 다음 단계: Berlekamp-Massey 알고리즘

신드롬을 계산했으니, 이제 이 신드롬으로부터 **오류 위치 다항식**을 찾아야 합니다.

→ `berlekamp_massey_learning.ipynb`로 계속

### BCH(7,4) 핵심 관찰

#### 단일 오류의 패턴

위치 j에 단일 오류가 있으면:
- S₁ = α^j
- S₂ = α^(2j) = (S₁)²

이 관계는 **단일 오류의 특징**입니다. 만약 S₂ ≠ (S₁)²이면 다중 오류가 있다는 것을 의미합니다.

#### 작은 예제의 장점

BCH(7,4)는 손으로 계산할 수 있을 만큼 작아서:
- 모든 GF(2³) 원소를 테이블로 볼 수 있음
- 각 단계를 구체적으로 확인 가능
- 더 큰 코드의 동작을 이해하는 기반

#### 다음 단계

이제 이 신드롬을 사용하여 **Berlekamp-Massey 알고리즘**으로 오류 위치를 찾아봅시다!

→ `berlekamp_massey_learning.ipynb`의 BCH(7,4) 섹션 참조

In [ ]:
# 예제 3: 위치 0에 오류
print("\n" + "=" * 80)
print("BCH(7,4) 예제 3: 위치 0에 오류")
print("=" * 80)

received_74_3 = [1, 0, 0, 0, 0, 0, 0]
print(f"수신: {''.join(map(str, received_74_3))}")
print(f"오류 위치: 0")

calc_74_3 = SyndromeCalculator(gf3, t=1, verbose=False)
result_74_3 = calc_74_3.compute_by_evaluation(received_74_3)

# 예상값
expected_s1_3 = gf3.one()
expected_s2_3 = gf3.one()

print(f"\n신드롬:")
print(f"  S₁ = {result_74_3.syndromes[0]}")
print(f"  S₂ = {result_74_3.syndromes[1]}")
print(f"\n검증:")
print(f"  예상 S₁ = 1 = {expected_s1_3} ✓" if result_74_3.syndromes[0] == expected_s1_3 else f"  예상 S₁ = 1 = {expected_s1_3} ✗")
print(f"  예상 S₂ = 1 = {expected_s2_3} ✓" if result_74_3.syndromes[1] == expected_s2_3 else f"  예상 S₂ = 1 = {expected_s2_3} ✗")
print(f"\n특징: 위치 0의 오류는 모든 신드롬이 1이 됩니다.")

### 예제 3: 위치 0에 오류

수신: `1000000`

예상 신드롬:
- S₁ = α⁰ = 1
- S₂ = α⁰ = 1

위치 0의 오류는 특별합니다: α⁰ = 1이므로 모든 신드롬이 1이 됩니다.

In [ ]:
# 예제 2: 위치 2에 오류
print("\n" + "=" * 80)
print("BCH(7,4) 예제 2: 위치 2에 오류")
print("=" * 80)

received_74_2 = [0, 0, 1, 0, 0, 0, 0]
print(f"수신: {''.join(map(str, received_74_2))}")
print(f"오류 위치: 2")

calc_74_2 = SyndromeCalculator(gf3, t=1, verbose=False)
result_74_2 = calc_74_2.compute_by_evaluation(received_74_2)

# 예상값
expected_s1_2 = gf3.alpha(2)
expected_s2_2 = gf3.alpha(4)

print(f"\n신드롬:")
print(f"  S₁ = {result_74_2.syndromes[0]}")
print(f"  S₂ = {result_74_2.syndromes[1]}")
print(f"\n검증:")
print(f"  예상 S₁ = α² = {expected_s1_2} ✓" if result_74_2.syndromes[0] == expected_s1_2 else f"  예상 S₁ = α² = {expected_s1_2} ✗")
print(f"  예상 S₂ = α⁴ = {expected_s2_2} ✓" if result_74_2.syndromes[1] == expected_s2_2 else f"  예상 S₂ = α⁴ = {expected_s2_2} ✗")
print(f"  S₂ = (S₁)²? {result_74_2.syndromes[1] == result_74_2.syndromes[0] ** 2} ✓")

### 예제 2: 위치 2에 오류

수신: `0010000`

예상 신드롬:
- S₁ = α²
- S₂ = α⁴

In [ ]:
# 예제 1: 위치 5에 오류
print("=" * 80)
print("BCH(7,4) 예제 1: 위치 5에 오류")
print("=" * 80)

received_74_1 = [0, 0, 0, 0, 0, 1, 0]
print(f"수신: {''.join(map(str, received_74_1))}")
print(f"오류 위치: 5")

# 신드롬 계산 (t=1이므로 S_1, S_2)
calc_74 = SyndromeCalculator(gf3, t=1, verbose=True)
result_74_1 = calc_74.compute_by_evaluation(received_74_1)

# 예상값 확인
expected_s1 = gf3.alpha(5)
expected_s2 = gf3.alpha(3)  # α^10 = α^3 (mod 7)

print(f"\n검증:")
print(f"  예상 S₁ = α⁵ = {expected_s1}")
print(f"  실제 S₁ = {result_74_1.syndromes[0]}")
print(f"  일치: {result_74_1.syndromes[0] == expected_s1}")
print()
print(f"  예상 S₂ = α³ = {expected_s2}")
print(f"  실제 S₂ = {result_74_1.syndromes[1]}")
print(f"  일치: {result_74_1.syndromes[1] == expected_s2}")
print()
print(f"  S₂ = (S₁)²? {result_74_1.syndromes[1] == result_74_1.syndromes[0] ** 2}")
print(f"  (α⁵)² = α¹⁰ = α³ ✓")

### 예제 1: 위치 5에 오류

#### 수신 벡터
```
r = (0, 0, 0, 0, 0, 1, 0)
```

인덱싱: r = (r₀, r₁, r₂, r₃, r₄, r₅, r₆)

오류는 r₅ = 1에만 있습니다.

#### 다항식 표현
```
R(x) = r₀ + r₁·x + r₂·x² + r₃·x³ + r₄·x⁴ + r₅·x⁵ + r₆·x⁶
     = 0 + 0·x + 0·x² + 0·x³ + 0·x⁴ + 1·x⁵ + 0·x⁶
     = x⁵
```

#### 예상 신드롬

오류 위치가 5이므로:
- S₁ = R(α) = α⁵
- S₂ = R(α²) = (α²)⁵ = α¹⁰

α¹⁰을 간단히 하면: α¹⁰ = α¹⁰ mod 7 = α³

따라서 예상:
- S₁ = α⁵
- S₂ = α³

In [ ]:
# GF(2^3) 생성 및 테이블 출력
gf3 = GaloisField(3)
print(f"생성된 유한체: {gf3}")
print(f"원시 다항식: {bin(gf3.primitive_poly)} (x³ + x + 1)")
print()

# 완전한 GF(2^3) 참조 테이블
create_gf_reference_table(gf3)

## 13. BCH(7,4) 완전 예제

이제 작고 구체적인 BCH(7,4) 예제를 통해 신드롬 계산의 모든 단계를 손으로 따라가볼 수 있습니다.

### BCH(7,4,1) 코드

- **n = 7**: 코드워드 길이
- **k = 4**: 정보 비트
- **t = 1**: 1비트 오류 정정
- **GF(2³)**: 원시 다항식 x³ + x + 1

### GF(2³) 완전 테이블

먼저 GF(2³)의 모든 원소를 확인합니다.